# Generation Evaluation

This notebook is the complete, chronological evaluation record for the RAG generation stage.

It documents what was done from the beginning of Step 10 through the final semantic analysis, while keeping generated experiment artifacts separate from analysis.

Canonical Step 10 flow:

`generation_evaluation.jsonl`
→ source-level retrieval verification
→ semantic evaluation rubric
→ Qwen3-8B semantic judge
→ `generation_semantic_judgments.jsonl` + `generation_semantic_review.csv`
→ failure/confidence analysis
→ engineering conclusions

Reproducibility rule: this notebook does not regenerate or overwrite completed experiment artifacts. Expensive/destructive commands are documented, not executed automatically.

## 1. Step 10 objective and fixed system

Step 10 evaluates the end-to-end RAG system selected in Step 09:

`question → PubMedBERT retrieval (top 20) → biomedical reranker → top 5 context → Qwen3-8B generation`

The generation model runs locally through Ollama.

Step 09 established the retrieval/reranking operating point, so Step 10 keeps that configuration fixed and asks whether the resulting context supports good answers and appropriate abstention.

## 2. Evaluation design

The generation evaluation contains 100 examples selected from the 500-example PubMedQA project corpus.

For each example, `generation_evaluation.jsonl` stores:

- PubMed ID
- question
- PubMedQA reference answer and decision
- generated answer
- retrieved/reranked context

Two different questions are evaluated:

1. Did retrieval find the correct source article?
2. Did the generator produce a correct, evidence-grounded answer?

A correct PubMed ID match is only a source-level retrieval diagnostic. It does not prove that the retrieved chunk contains the answer-bearing evidence.

Known limitations include the same-corpus evaluation design, the 100-example size, and model-assisted semantic judging.

In [7]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("..")
DATA = ROOT / "data"

GEN_PATH = DATA / "generation_evaluation.jsonl"
SEMANTIC_PATH = DATA / "generation_semantic_review.csv"
SEMANTIC_PRESERVED_COPY = DATA / "generation_semantic_review_100_qwen3-8b.csv"

assert GEN_PATH.exists(), f"Missing: {GEN_PATH}"
assert SEMANTIC_PATH.exists(), f"Missing: {SEMANTIC_PATH}"

print("Canonical raw generation artifact:", GEN_PATH)
print("Canonical semantic-judge CSV:", SEMANTIC_PATH)

if SEMANTIC_PRESERVED_COPY.exists():
    print("Preserved experiment copy also present:", SEMANTIC_PRESERVED_COPY)
    print("It is not treated as a separate experiment unless its contents differ.")

Canonical raw generation artifact: ../data/generation_evaluation.jsonl
Canonical semantic-judge CSV: ../data/generation_semantic_review.csv
Preserved experiment copy also present: ../data/generation_semantic_review_100_qwen3-8b.csv
It is not treated as a separate experiment unless its contents differ.


In [8]:
# Non-destructive preflight
# This notebook never writes to Step 10 experiment artifacts.
PROTECTED_ARTIFACTS = [
    GEN_PATH,
    SEMANTIC_PATH,
    DATA / "generation_semantic_judgments.jsonl",
    SEMANTIC_PRESERVED_COPY,
]

for path in PROTECTED_ARTIFACTS:
    if path.exists():
        print(f"PROTECTED — will not overwrite: {path}")
    else:
        print(f"Not present: {path}")

print("\nNo notebook cell below writes to any protected Step 10 artifact.")

PROTECTED — will not overwrite: ../data/generation_evaluation.jsonl
PROTECTED — will not overwrite: ../data/generation_semantic_review.csv
PROTECTED — will not overwrite: ../data/generation_semantic_judgments.jsonl
PROTECTED — will not overwrite: ../data/generation_semantic_review_100_qwen3-8b.csv

No notebook cell below writes to any protected Step 10 artifact.


## 3. Artifact provenance

The intended provenance is:

`evaluate_generation.py`
→ `data/generation_evaluation.jsonl`

Then:

`judge_generation.py`
→ `data/generation_semantic_judgments.jsonl`
→ `data/generation_semantic_review.csv`

The file `generation_semantic_review_100_qwen3-8b.csv` is a preserved copy of the completed 100-example semantic review, created later so the completed result would have an experiment-specific name. It is not a second judging run.

The blank/manual-review template that existed during earlier notebook iterations is an intermediate workflow artifact, not the source of the final Qwen3-8B judgements. It is not recreated here.

## 4. Load the immutable generation output

`generation_evaluation.jsonl` is the primary Step 10 generation artifact. Analysis reads it; it should not be overwritten during notebook execution.

In [9]:
records = []
with GEN_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

generation_df = pd.DataFrame(records)

print(f"Examples: {len(generation_df)}")
print("Columns:", list(generation_df.columns))
assert len(generation_df) == 100

generation_df.head(2)

Examples: 100
Columns: ['pubid', 'question', 'reference_answer', 'reference_decision', 'generated_answer', 'retrieved_context']


,pubid,question,reference_answer,reference_decision,generated_answer,retrieved_context
0,10331115,Is amoxapine an atypical antipsychotic?,PET data show that amoxapine's profile is very...,yes,Amoxapine is considered an atypical antipsycho...,"[{'rank': 1, 'pubid': '10331115', 'chunk_index..."
1,10749257,Has the mammography quality standards act affe...,"In 1995, the first year that the MQSA was in e...",yes,"Yes, the Mammography Quality Standards Act (MQ...","[{'rank': 1, 'pubid': '10749257', 'chunk_index..."


## 5. Reconstruct source-level retrieval metrics

The raw JSONL does not store `source_rank` as a top-level field. The rank is reconstructed from `retrieved_context` by finding the first retrieved item whose PubMed ID matches the example's target PubMed ID.

This is exactly the source-level diagnostic later written into the semantic-review CSV by `judge_generation.py`.

In [10]:
def source_rank_from_record(record):
    matches = [
        int(item["rank"])
        for item in record.get("retrieved_context", [])
        if str(item.get("pubid")) == str(record.get("pubid"))
    ]
    return matches[0] if matches else None

generation_df["source_rank"] = generation_df.apply(
    lambda row: source_rank_from_record(row.to_dict()), axis=1
)
generation_df["source_in_top5"] = generation_df["source_rank"].notna()

print(
    f"Correct source at reranked rank 1: "
    f"{(generation_df['source_rank'] == 1).sum()}/{len(generation_df)} "
    f"({(generation_df['source_rank'] == 1).mean():.1%})"
)
print(
    f"Correct source in reranked top 5: "
    f"{generation_df['source_in_top5'].sum()}/{len(generation_df)} "
    f"({generation_df['source_in_top5'].mean():.1%})"
)

generation_df["source_rank"].value_counts().sort_index()

Correct source at reranked rank 1: 99/100 (99.0%)
Correct source in reranked top 5: 100/100 (100.0%)


source_rank
1    99
2     1
Name: count, dtype: int64

### Result

For the 100-example generation set:

- 99/100 (99%) had the correct source at reranked rank 1.
- 100/100 (100%) had the correct source somewhere in the reranked top 5.

Again, these are source-level retrieval figures, not answer-quality scores.

## 6. Why lexical/source matching is not enough

A source-ID match answers: “Did the system retrieve a chunk from the article associated with this question?”

It does not answer: “Did the supplied context contain the information necessary to answer this question?”

This distinction explains why Step 10 needs semantic evaluation. A system can retrieve the right article, pass the source-level metric, and still fail because the selected passage is indirect, incomplete, or irrelevant to the specific question.

## 7. Semantic evaluation rubric

The semantic judge assigns:

**Answer correctness**
- `correct`
- `partially_correct`
- `incorrect`
- `unclear`

**Evidence grounding**
- `grounded`
- `partially_grounded`
- `unsupported`
- `unclear`

**Abstention behaviour**
- `appropriate_abstention`
- `answered_with_insufficient_evidence`
- `over_abstention`
- `not_applicable`

**Failure attribution**
- `no_failure`
- `generation_failure`
- `retrieval_failure`

**Confidence**
- `high`
- `medium`
- `low`

The judge is instructed to use only the question, reference answer/decision, retrieved context, and generated answer. It must not fill missing evidence from outside biomedical knowledge.

## 8. Historical manual-review/template step

Earlier in the Stage 10 development process, a blank semantic-review CSV was created as an intermediate scaffold for reviewing the generated examples. That was useful as a workflow template, but it is not the completed evaluation result.

The final judging workflow became:

`generation_evaluation.jsonl`
→ `judge_generation.py`
→ semantic judgments
→ completed `generation_semantic_review.csv`

The blank template is therefore historical documentation only. This notebook does not recreate it and cannot overwrite the completed CSV.

## 9. Qwen3-8B semantic judge

The completed semantic labels were generated using the local `qwen3:8b` model through Ollama.

The judge receives the question, reference answer/decision, retrieved context, and generated answer. It returns structured semantic labels plus a reason.

Important limitation: the same model family was used for generation and judging. Therefore these labels are model-assisted evaluation, not independent human ground truth, and shared-model bias is possible.

### Historical execution command and safe rerun protocol

The original semantic judging run was executed outside this notebook:

```bash
python src/judge_generation.py
```

That run produced:

```text
data/generation_semantic_judgments.jsonl
data/generation_semantic_review.csv
```

For the **canonical 100-example experiment, do not run the command again**. The completed outputs are preserved and this notebook only consumes them.

The canonical judging workflow must be non-destructive: a future run must refuse to overwrite an existing completed result. A new run must use an explicitly different output location/name.

For example:

```bash
python src/judge_generation.py     --output-jsonl data/generation_semantic_judgments_v2.jsonl     --output-csv data/generation_semantic_review_v2.csv
```

If those output files already exist, the script must refuse to overwrite them.

This makes the artifact chain explicit:

`generation_evaluation.jsonl`
→ `judge_generation.py`
→ `generation_semantic_judgments.jsonl`
→ `generation_semantic_review.csv`

The experiment-specific preserved copy `generation_semantic_review_100_qwen3-8b.csv` is a copy of the completed CSV, not a second judging run.

### Safe judge execution contract

The existing completed `generation_semantic_review.csv` must remain untouched.

Before running a future judging experiment, use new output paths. The judge should enforce this with an existence check equivalent to:

```python
if output_jsonl.exists() or output_csv.exists():
    raise FileExistsError("Refusing to overwrite existing experiment artifacts.")
```

For a new experiment, use explicit paths, for example:

```bash
python src/judge_generation.py --output-jsonl data/generation_semantic_judgments_v2.jsonl --output-csv data/generation_semantic_review_v2.csv
```

Do not use a new run to replace the canonical 100-example result.

Because the current runtime does not contain a writable copy of `src/judge_generation.py`, this notebook does not pretend to modify that external script. The canonical completed results remain preserved.

## 10. Load the completed semantic review

This is the completed 100-example semantic result used for the remainder of the analysis.

In [11]:
semantic_df = pd.read_csv(SEMANTIC_PATH)

required_cols = [
    "pubid", "question", "reference_decision", "reference_answer",
    "generated_answer", "source_rank", "source_in_top5",
    "answer_correctness", "evidence_grounding",
    "abstention_behavior", "failure_attribution",
    "confidence", "review_notes"
]
missing = [c for c in required_cols if c not in semantic_df.columns]
assert not missing, f"Missing columns: {missing}"
assert len(semantic_df) == 100

print(f"Loaded {len(semantic_df)} completed semantic judgements.")
semantic_df.head(2)

Loaded 100 completed semantic judgements.


,pubid,question,reference_decision,reference_answer,generated_answer,source_rank,source_in_top5,answer_correctness,evidence_grounding,abstention_behavior,failure_attribution,confidence,review_notes
0,10331115,Is amoxapine an atypical antipsychotic?,yes,PET data show that amoxapine's profile is very...,Amoxapine is considered an atypical antipsycho...,1,True,correct,grounded,not_applicable,no_failure,high,The generated answer correctly identifies amox...
1,10749257,Has the mammography quality standards act affe...,yes,"In 1995, the first year that the MQSA was in e...","Yes, the Mammography Quality Standards Act (MQ...",1,True,correct,grounded,appropriate_abstention,no_failure,high,The generated answer correctly cites PubMed ID...


## 11. Sanity-check the semantic CSV against the raw generation artifact

The semantic CSV should correspond one-to-one with the raw generation evaluation. This check verifies PubMed IDs and generated answers rather than assuming the files are aligned.

In [12]:
raw_by_id = generation_df.set_index(generation_df["pubid"].astype(str))
sem_by_id = semantic_df.set_index(semantic_df["pubid"].astype(str))

assert set(raw_by_id.index) == set(sem_by_id.index)

mismatched_answers = []
for pubid in raw_by_id.index:
    raw_answer = str(raw_by_id.loc[pubid, "generated_answer"])
    sem_answer = str(sem_by_id.loc[pubid, "generated_answer"])
    if raw_answer != sem_answer:
        mismatched_answers.append(pubid)

print("PubMed ID sets match:", True)
print("Generated-answer mismatches:", len(mismatched_answers))
assert not mismatched_answers

PubMed ID sets match: True
Generated-answer mismatches: 0


## 12. Aggregate semantic results

In [13]:
for col in [
    "answer_correctness",
    "evidence_grounding",
    "abstention_behavior",
    "failure_attribution",
    "confidence",
]:
    print(f"\n{col}")
    print(semantic_df[col].value_counts(dropna=False).to_string())


answer_correctness
answer_correctness
correct              89
incorrect             6
partially_correct     5

evidence_grounding
evidence_grounding
grounded              95
partially_grounded     4
unsupported            1

abstention_behavior
abstention_behavior
appropriate_abstention                 53
not_applicable                         39
answered_with_insufficient_evidence     5
over_abstention                         3

failure_attribution
failure_attribution
no_failure            85
generation_failure     9
retrieval_failure      6

confidence
confidence
high      90
medium    10


### Completed 100-example results

The preserved semantic review contains:

- 89% correct, 5% partially correct, 6% incorrect.
- 95% grounded, 4% partially grounded, 1% unsupported.
- 53% appropriate abstention, 5% answered with insufficient evidence, 3% over-abstention, and 39% not applicable.
- 85% no failure, 9% generation failure, 6% retrieval failure.
- 90% high confidence and 10% medium confidence.

These figures are Qwen3-8B judge outputs, not human-annotated ground truth.

## 13. Inspect individual cases

Aggregate scores are not enough for an engineering evaluation. The helper below allows the actual question, reference, generated answer, and semantic judgement to be inspected for any PubMed ID.

In [14]:
pd.set_option("display.max_colwidth", 120)

def show_case(pubid):
    row = semantic_df[semantic_df["pubid"].astype(str) == str(pubid)]
    if row.empty:
        raise ValueError(f"PubMed ID {pubid} not found")
    r = row.iloc[0]

    print("=" * 100)
    print("PubMed ID:", r["pubid"])
    print("\nQUESTION:\n", r["question"])
    print("\nREFERENCE DECISION:\n", r["reference_decision"])
    print("\nREFERENCE ANSWER:\n", r["reference_answer"])
    print("\nGENERATED ANSWER:\n", r["generated_answer"])
    print("\nJUDGEMENT:")
    for c in [
        "answer_correctness", "evidence_grounding",
        "abstention_behavior", "failure_attribution", "confidence"
    ]:
        print(f"  {c}: {r[c]}")
    print("\nREVIEW NOTES:\n", r["review_notes"])

# Example:
# show_case(10331115)

## 14. Representative-case selection

The completed review should be inspected for at least four patterns:

1. correct + grounded answer;
2. appropriate abstention when evidence is insufficient;
3. generation failure despite useful evidence being present;
4. retrieval failure where answer-bearing evidence is unavailable.

The next cell automatically selects one available example for each category where possible, avoiding hard-coded PubMed IDs.

In [15]:
def first_matching(mask):
    matches = semantic_df.loc[mask, "pubid"]
    return matches.iloc[0] if not matches.empty else None

representatives = {
    "correct_and_grounded": first_matching(
        (semantic_df["answer_correctness"] == "correct") &
        (semantic_df["evidence_grounding"] == "grounded")
    ),
    "appropriate_abstention": first_matching(
        semantic_df["abstention_behavior"] == "appropriate_abstention"
    ),
    "generation_failure": first_matching(
        semantic_df["failure_attribution"] == "generation_failure"
    ),
    "retrieval_failure": first_matching(
        semantic_df["failure_attribution"] == "retrieval_failure"
    ),
}

representatives

{'correct_and_grounded': np.int64(10331115),
 'appropriate_abstention': np.int64(10749257),
 'generation_failure': np.int64(12407608),
 'retrieval_failure': np.int64(11867487)}

## 15. Step 09 smoke-test connection

The Step 09 smoke test already demonstrated an important property: when the retrieved context was weak or indirect, Qwen3-8B could abstain instead of inventing an answer.

Step 10 turns that qualitative observation into a systematic evaluation dimension. It also exposes the complementary problem: appropriate abstention is useful, but over-abstention can occur when sufficient evidence is actually present.

## 16. Inspect all attributed failures

The completed semantic review identifies 15 attributed failures:

- 9 generation failures;
- 6 retrieval failures.

The table below is generated directly from the preserved CSV so the failure list is reproducible rather than manually copied into the notebook.

In [16]:
failures = semantic_df[
    semantic_df["failure_attribution"] != "no_failure"
].copy()

print("Attributed failures:", len(failures))
print(failures["failure_attribution"].value_counts().to_string())

failures[[
    "pubid", "question", "answer_correctness",
    "evidence_grounding", "abstention_behavior",
    "failure_attribution", "confidence", "review_notes"
]].sort_values(["failure_attribution", "pubid"])

Attributed failures: 15
failure_attribution
generation_failure    9
retrieval_failure     6


,pubid,question,answer_correctness,evidence_grounding,abstention_behavior,failure_attribution,confidence,review_notes
12,12407608,Does ultrasound imaging before puncture facilitate internal jugular vein cannulation?,incorrect,partially_grounded,answered_with_insufficient_evidence,generation_failure,medium,"The generated answer incorrectly generalizes the study's findings, claiming ultrasound is universally beneficial. Th..."
15,12846929,Quality of life in lung cancer patients: does socioeconomic status matter?,partially_correct,partially_grounded,answered_with_insufficient_evidence,generation_failure,medium,The generated answer correctly notes baseline SES impact but incorrectly cites a study that does not address SES as ...
24,16776337,Pituitary apoplexy: do histological features influence the clinical presentation and outcome?,incorrect,partially_grounded,over_abstention,generation_failure,medium,"The generated answer incorrectly claims the context lacks results from PubMed ID 16776337, but the retrieved context..."
25,16778275,Is routine chest radiography after transbronchial biopsy necessary?,incorrect,unsupported,answered_with_insufficient_evidence,generation_failure,medium,"The generated answer fails to address the question directly and does not engage with the provided context, which con..."
35,17919952,Are reports of mechanical dysfunction in chronic oro-facial pain related to somatisation?,incorrect,grounded,over_abstention,generation_failure,medium,The generated answer contradicts the reference answer by asserting mechanical factors are not directly related to so...
46,19155657,Does accompanying metabolic syndrome contribute to heart dimensions in hypertensive patients?,partially_correct,grounded,answered_with_insufficient_evidence,generation_failure,medium,"The generated answer correctly identifies left atrial diameter as influenced by MetS, supported by the context. Howe..."
66,23690198,Implementation of epidural analgesia for labor: is the standard of effective analgesia reachable in all women?,incorrect,partially_grounded,over_abstention,generation_failure,medium,"The generated answer incorrectly states that epidural analgesia is not universally reachable, conflicting with the r..."
76,24977765,Are pediatric concussion patients compliant with discharge instructions?,partially_correct,grounded,not_applicable,generation_failure,medium,The generated answer correctly references the 39% RTP rate from the context but does not explicitly align with the r...
91,27581329,Estimation of basal metabolic rate in Chinese: are the current prediction equations applicable?,incorrect,grounded,answered_with_insufficient_evidence,generation_failure,medium,The generated answer contradicts the PubMedQA reference answer by concluding current equations may not be fully appl...
8,11867487,Does rugby headgear prevent concussion?,correct,grounded,appropriate_abstention,retrieval_failure,high,The generated answer correctly identifies that the retrieved context only contains subjective attitudes (62% of play...


## 17. Inspect the 10 medium-confidence cases

The completed review has 10 medium-confidence cases and no low-confidence cases. These are priority cases for independent human review if the project is later upgraded from a portfolio evaluation to a stronger benchmark.

In [17]:
uncertain = semantic_df[
    semantic_df["confidence"] != "high"
].copy()

print("Medium/low-confidence cases:", len(uncertain))
print(uncertain["confidence"].value_counts().to_string())

uncertain[[
    "pubid", "question", "answer_correctness",
    "evidence_grounding", "abstention_behavior",
    "failure_attribution", "confidence", "review_notes"
]].sort_values(["confidence", "pubid"])

Medium/low-confidence cases: 10
confidence
medium    10


,pubid,question,answer_correctness,evidence_grounding,abstention_behavior,failure_attribution,confidence,review_notes
12,12407608,Does ultrasound imaging before puncture facilitate internal jugular vein cannulation?,incorrect,partially_grounded,answered_with_insufficient_evidence,generation_failure,medium,"The generated answer incorrectly generalizes the study's findings, claiming ultrasound is universally beneficial. Th..."
15,12846929,Quality of life in lung cancer patients: does socioeconomic status matter?,partially_correct,partially_grounded,answered_with_insufficient_evidence,generation_failure,medium,The generated answer correctly notes baseline SES impact but incorrectly cites a study that does not address SES as ...
24,16776337,Pituitary apoplexy: do histological features influence the clinical presentation and outcome?,incorrect,partially_grounded,over_abstention,generation_failure,medium,"The generated answer incorrectly claims the context lacks results from PubMed ID 16776337, but the retrieved context..."
25,16778275,Is routine chest radiography after transbronchial biopsy necessary?,incorrect,unsupported,answered_with_insufficient_evidence,generation_failure,medium,"The generated answer fails to address the question directly and does not engage with the provided context, which con..."
35,17919952,Are reports of mechanical dysfunction in chronic oro-facial pain related to somatisation?,incorrect,grounded,over_abstention,generation_failure,medium,The generated answer contradicts the reference answer by asserting mechanical factors are not directly related to so...
46,19155657,Does accompanying metabolic syndrome contribute to heart dimensions in hypertensive patients?,partially_correct,grounded,answered_with_insufficient_evidence,generation_failure,medium,"The generated answer correctly identifies left atrial diameter as influenced by MetS, supported by the context. Howe..."
66,23690198,Implementation of epidural analgesia for labor: is the standard of effective analgesia reachable in all women?,incorrect,partially_grounded,over_abstention,generation_failure,medium,"The generated answer incorrectly states that epidural analgesia is not universally reachable, conflicting with the r..."
76,24977765,Are pediatric concussion patients compliant with discharge instructions?,partially_correct,grounded,not_applicable,generation_failure,medium,The generated answer correctly references the 39% RTP rate from the context but does not explicitly align with the r...
91,27581329,Estimation of basal metabolic rate in Chinese: are the current prediction equations applicable?,incorrect,grounded,answered_with_insufficient_evidence,generation_failure,medium,The generated answer contradicts the PubMedQA reference answer by concluding current equations may not be fully appl...
93,28359277,Do healthier lifestyles lead to less utilization of healthcare resources?,partially_correct,grounded,appropriate_abstention,retrieval_failure,medium,The generated answer correctly identifies that the retrieved context shows a significant correlation between health ...


## 18. Failure-attribution matrix

In [18]:
pd.crosstab(
    semantic_df["failure_attribution"],
    semantic_df["answer_correctness"],
    margins=True
)

answer_correctness,correct,incorrect,partially_correct,All
failure_attribution,,,,
generation_failure,0,6,3,9
no_failure,85,0,0,85
retrieval_failure,4,0,2,6
All,89,6,5,100


In [19]:
pd.crosstab(
    semantic_df["failure_attribution"],
    semantic_df["evidence_grounding"],
    margins=True
)

evidence_grounding,grounded,partially_grounded,unsupported,All
failure_attribution,,,,
generation_failure,4,4,1,9
no_failure,85,0,0,85
retrieval_failure,6,0,0,6
All,95,4,1,100


## 19. Engineering interpretation

The evaluation shows strong source-level retrieval but a non-zero end-to-end failure rate.

The 99% rank-1 source retrieval figure means the system usually finds the correct article. The six retrieval-attributed failures show why article-level retrieval is not enough: the selected context can still fail to expose the evidence needed for the specific question.

The nine generation-attributed failures show a separate optimisation target. Even with useful context available, the generator can fail through incomplete answers, inappropriate abstention, or other answer-construction errors.

The appropriate engineering loop is therefore:

`retrieve → rerank → verify evidence sufficiency → generate → semantically evaluate → classify failure → improve the responsible stage`

This is a stronger production-oriented conclusion than simply reporting one generation accuracy number.

## 20. Reproducibility and non-overwrite policy

Canonical Step 10 artifacts:

```text
src/evaluate_generation.py
    produces
data/generation_evaluation.jsonl

src/judge_generation.py
    produces
data/generation_semantic_judgments.jsonl
data/generation_semantic_review.csv

notebooks/10_generation_evaluation.ipynb
    consumes the preserved artifacts and performs analysis
```

The experiment-specific copy:

```text
data/generation_semantic_review_100_qwen3-8b.csv
```

is a preservation copy of the completed 100-example semantic CSV, not a new judging experiment.

The notebook does not write to any of these files.

The judging script is also required to be non-destructive: it must refuse to overwrite an existing output file. New experiments must use explicitly named output paths.

This gives every generated artifact a clear provenance and prevents accidental loss of prior experimental results.

## 21. Limitations

1. The 100 questions come from the project corpus, so this is not a clean independent held-out benchmark.
2. Source-level PubMed-ID retrieval is weaker than answer-bearing context evaluation.
3. Semantic labels are model-assisted rather than human ground truth.
4. The generator and judge use the same Qwen3-8B model family, creating possible shared-model bias.
5. PubMedQA reference answers are concise dataset references and do not define every acceptable formulation.
6. One hundred examples are useful for engineering diagnosis but insufficient for strong clinical-quality claims.

## 22. Final Step 10 conclusion

The RAG system demonstrates strong source-level retrieval and generally strong grounded generation on this 100-example evaluation. The most valuable result is the failure decomposition:

- retrieval failures identify cases where the pipeline did not expose sufficient answer-bearing evidence;
- generation failures identify cases where the model did not make appropriate use of the available evidence;
- abstention labels expose the trade-off between avoiding unsupported claims and over-abstaining.

Step 10 therefore produces an actionable engineering backlog rather than only a headline score.

The next iteration should focus on the actual 15 attributed failures, especially answer-bearing retrieval/context construction for retrieval failures and prompt/context/generation behaviour for generation failures. Independent human or alternative-model validation should be added before making stronger claims about semantic accuracy.

### Manual review of semantic-judge failure cases

The semantic judge identified 15 cases with an attributed failure: 9 generation failures and 6 retrieval failures. I reviewed these cases individually using the original question, PubMedQA reference answer, generated answer, and the actual retrieved context recorded in `generation_evaluation.jsonl`.

The purpose of this review was not to replace the semantic judge, but to check whether its failure attribution was reasonable and to identify recurring engineering failure modes.

Overall, the judge's 9-generation / 6-retrieval split is broadly reasonable. However, at least one case is borderline, which is an important limitation of using an LLM as an evaluator.

#### Retrieval-attributed failures

**11867487 — Retrieval failure: confirmed.**

The question asks whether rugby headgear prevents concussion. The retrieved context discusses attitudes toward wearing protective headgear and the relationship between attitudes and behaviour, but it does not provide evidence about whether headgear actually prevents concussion. The generated answer appropriately avoids making a definitive claim. This is a genuine evidence-retrieval/context-selection problem: the system found a related article/passage but not the answer-bearing evidence required by the question.

**17444776 — Retrieval failure: confirmed, with a caveat.**

The question requires a comparative judgement about which intervention is best. The retrieved context comes from the relevant source but does not provide enough comparative evidence to establish the requested conclusion. The model therefore lacks sufficient evidence to answer confidently. This illustrates the distinction between retrieving the correct source article and retrieving the specific evidence needed to answer the question.

**24160268 — Retrieval failure: confirmed.**

The retrieved evidence is not relevant enough to answer the question. The model appropriately abstains rather than inventing an answer. This is a clear retrieval/context failure rather than a generation failure.

**26134053 — Retrieval failure: confirmed.**

The retrieved context does not contain the specific evidence needed for the question, even though the correct source article is involved. The reference answer depends on a particular finding that is not adequately represented in the supplied context. The model therefore cannot reliably reconstruct the benchmark answer from the available evidence.

**27136599 — Retrieval failure: confirmed.**

The retrieved passage describes the study and indicates that leakage was measured, but it does not provide the actual result needed to answer the question about safety. The model's refusal to invent the result is appropriate. This is another example of correct-source retrieval without answer-bearing evidence.

**28359277 — Retrieval failure: confirmed.**

The retrieved context establishes an association between health behaviours and healthcare utilisation, but does not establish the direction of the relationship needed by the question. The model therefore cannot responsibly conclude that the behaviour is associated with less utilisation. This is a clear example of why source-level retrieval metrics are insufficient: the correct article can be retrieved while the selected passage still lacks the answer-bearing detail.

#### Generation-attributed failures

**12407608 — Generation failure: confirmed, but the error is better described as partial correctness/overgeneralisation.**

The generated answer captures an important conditional finding: ultrasound was useful in situations where respiratory venodilation could not be visualised, while outcomes were similar when it could be visualised. The problem is that the answer opens with a general statement that ultrasound facilitates the procedure, which overgeneralises the more conditional study finding. The answer therefore contains the core result but does not preserve the qualification accurately.

**12846929 — Generation failure: confirmed.**

The main answer correctly identifies the relationship between socioeconomic status and quality of life: lower socioeconomic status was associated with worse baseline quality of life, while the difference disappeared at follow-up. However, the generated response also introduces an additional claim about another study/prognostic factor that is not supported by the supplied context. This is a useful example of a mostly correct answer becoming unreliable because the model adds unsupported information.

**16776337 — Borderline between retrieval and generation failure.**

The judge classified this as a generation failure because relevant study information appears in the retrieved context. However, on inspection, the supplied passage primarily describes the study/question/design and does not clearly provide the actual outcome needed to answer the question. The model's statement that the evidence was insufficient is therefore understandable.

This case should be treated as a borderline retrieval/context failure rather than a clean generation failure. It demonstrates an important limitation of semantic failure attribution: determining whether a model “failed to use” evidence or correctly recognised that the evidence was incomplete can itself require human judgement.

**16778275 — Generation failure: confirmed.**

The supplied context contains answer-bearing information, but the model fails to provide the requested answer. This is a clearer generation-side failure because the necessary evidence is available in the context and the problem is the model's failure to use it appropriately.

**17919952 — Generation failure: mostly confirmed.**

The generated answer captures part of the relevant finding but mishandles the study's conclusion regarding somatisation. The issue is therefore not primarily absence of evidence; rather, the model interprets or expresses the evidence incorrectly. The judge's “incorrect” classification may be somewhat harsh because part of the answer is correct, but the case still represents a generation-side problem.

**19155657 — Generation failure: confirmed.**

The context contains the relevant finding, and the generated answer captures part of it but does not fully answer the broader question. The model extracts a relevant result but produces an incomplete answer. This is an important generation failure mode because retrieval succeeded but the final response does not fully satisfy the question.

**23690198 — Generation failure: confirmed.**

The supplied context contains evidence relevant to successful implementation/adherence. The model focuses on the reported 51% uptake figure but interprets it incorrectly instead of using the evidence to reach the benchmark conclusion. The failure is therefore in interpretation and answer construction rather than retrieval.

**24977765 — Generation failure: confirmed.**

The generated answer identifies a relevant numerical finding, including the 39% figure, but does not actually provide the conclusion requested by the question. The model therefore uses some of the evidence without converting it into the correct answer. This is an incomplete-answer generation failure.

**27581329 — Generation failure: confirmed.**

The relevant evidence is present in the supplied context, but the generated answer reaches an incorrect conclusion about its applicability. This is a clear generation-side interpretation failure: the model had access to relevant evidence but did not draw the supported conclusion.

### Overall assessment of the failure review

The manual review broadly supports the semantic judge's failure taxonomy. The six retrieval-attributed cases generally involve situations where the retrieved context does not contain sufficient answer-bearing evidence, even when the correct source article is present. The nine generation-attributed cases generally involve evidence being available but the model overgeneralising, adding unsupported information, failing to provide the complete answer, over-abstaining, or drawing an incorrect conclusion.

The main recurring generation failure modes are therefore:

1. overgeneralisation of a conditional finding;
2. unsupported additions beyond the retrieved evidence;
3. incomplete answers;
4. failure to use evidence that is present;
5. incorrect interpretation of otherwise relevant evidence;
6. over-abstention when the available evidence may be sufficient.

The main recurring retrieval failure mode is:

1. retrieving the correct or related source but failing to retrieve an answer-bearing passage.

This reinforces an important architectural conclusion: high source-level retrieval performance does not guarantee high answer quality. The system must retrieve the evidence needed for the specific question, not merely the correct article.

The review also identified one borderline case (16776337), where it is difficult to cleanly distinguish retrieval/context insufficiency from generation failure. This is an important limitation of model-assisted failure attribution and supports the use of independent human review for future, larger evaluations.

### Final Step 10 assessment

The Step 10 evaluation therefore provides evidence that the system is generally capable of producing grounded answers, but that remaining errors occur at both the retrieval/context and generation stages.

The strongest engineering findings are:

- source-level retrieval is very strong: 99% of examples had the correct source at reranked rank 1 and 100% had it in the top 5;
- semantic answer quality was judged as 89% correct, with 5% partially correct and 6% incorrect;
- 95% of answers were judged grounded;
- 9 cases were attributed to generation failure and 6 to retrieval failure;
- the generator generally shows useful evidence-boundary behaviour, but can also over-abstain;
- retrieving the correct article is not sufficient if the selected context does not contain the answer-bearing result;
- generation failures are primarily related to interpretation, completeness, overgeneralisation, unsupported additions, and evidence use.

The appropriate next improvement loop is therefore:

`retrieval → answer-bearing evidence → reranking/context construction → generation → semantic evaluation → failure attribution → targeted improvement`

Rather than immediately replacing the generation model, the failure analysis suggests that the next engineering work should target the specific failure modes identified above.